# ANN experiments with MLflow

Start the tracking server from the project root (`uv run mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5000`), then run this notebook. Compare the two ANN runs in the MLflow UI at `http://localhost:5000`.

In [ ]:
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
data_path = Path("winequality-white.csv")
if not data_path.exists():
    data_path = Path("2-dlproject") / "winequality-white.csv"

data = pd.read_csv(data_path, sep=";")
X = data.drop(columns="quality").to_numpy()
y = data["quality"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}")

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
experiment = mlflow.set_experiment("wine-quality-ann")

configs = [
    {"name": "small_ann", "hidden_units": 16, "learning_rate": 0.001},
    {"name": "wide_ann", "hidden_units": 64, "learning_rate": 0.001},
]

for config in configs:
    tf.keras.utils.set_random_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(config["hidden_units"], activation="relu"),
        tf.keras.layers.Dense(1),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss="mse",
        metrics=["mae"],
    )

    with mlflow.start_run(run_name=config["name"]):
        mlflow.log_params({
            "hidden_units": config["hidden_units"],
            "learning_rate": config["learning_rate"],
            "epochs": 20,
            "batch_size": 32,
        })
        model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
        test_mse, test_mae = model.evaluate(X_test, y_test, verbose=0)
        mlflow.log_metrics({
            "test_rmse": float(np.sqrt(test_mse)),
            "test_mae": float(test_mae),
        })
        print(f"{config['name']}: RMSE={np.sqrt(test_mse):.3f}, MAE={test_mae:.3f}")

In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_rmse ASC"],
)
runs[[
    "tags.mlflow.runName",
    "params.hidden_units",
    "params.learning_rate",
    "metrics.test_rmse",
    "metrics.test_mae",
]]